In [15]:
import pandas as pd

# Mapping Regions to States

Right now, the process we are using to map regions to states is fairly rudimentary.

Using the information provided by LBNL, we find which states are in which regions, and then set the state cost equal to the average cost of the regions it falls in. 

The first step is to make a dataframe with a row for each state, year, and region pairing. Alaska is excluded because it is excluded from LBNL datasets

In [24]:
region_mappings = {
    'CA': ['CAISO'],
    'AZ': ['West (non-ISO)'],
    'CO': ['West (non-ISO)'],
    'ID': ['West (non-ISO)'],
    'MT': ['West (non-ISO)', 'MISO', 'SPP'],
    'NV': ['West (non-ISO)'],
    'NM': ['West (non-ISO)', 'SPP'],
    'OR': ['West (non-ISO)'],
    'UT': ['West (non-ISO)'],
    'WA': ['West (non-ISO)'],
    'WY': ['West (non-ISO)'],
    'AR': ['MISO', 'SPP'],
    'IL': ['MISO', 'PJM'],
    'IN': ['MISO', 'PJM'],
    'IA': ['MISO', 'SPP'],
    'KY': ['MISO', 'PJM'],
    'LA': ['MISO', 'SPP'],
    'MI': ['MISO', 'PJM'],
    'MN': ['MISO', 'SPP'],
    'MS': ['MISO', 'Southeast (non-ISO)'],
    'MO': ['MISO', 'SPP'],
    'ND': ['MISO', 'SPP'],
    'SD': ['MISO', 'SPP'],
    'TX': ['MISO', 'SPP', 'ERCOT'],
    'WI': ['MISO'],
    'KS': ['SPP'],
    'NE': ['SPP'],
    'OK': ['SPP'],
    'DE': ['PJM'],
    'MD': ['PJM'],
    'NJ': ['PJM'],
    'NC': ['PJM', 'Southeast (non-ISO)'],
    'OH': ['PJM'],
    'PA': ['PJM'],
    'TN': ['PJM', 'Southeast (non-ISO)'],
    'VA': ['PJM'],
    'WV': ['PJM'],
    'DC': ['PJM'],
    'NY': ['NYISO'],
    'CT': ['ISO-NE'],
    'ME': ['ISO-NE'],
    'MA': ['ISO-NE'],
    'NH': ['ISO-NE'],
    'RI': ['ISO-NE'],
    'VT': ['ISO-NE'],
    'AL': ['Southeast (non-ISO)'],
    'FL': ['Southeast (non-ISO)'],
    'GA': ['Southeast (non-ISO)'],
    'SC': ['Southeast (non-ISO)'],
    'HI': ['Hawaii']
}

years = range(2015, 2025)
data = []

for state, regions in region_mappings.items():
    for region in regions:
        for year in years:
            data.append({
                'State': state,
                'Region': region,
                'Year': year
            })

df = pd.DataFrame(data)
df.head()

,State,Region,Year
0,CA,CAISO,2015
1,CA,CAISO,2016
2,CA,CAISO,2017
3,CA,CAISO,2018
4,CA,CAISO,2019


# Adding solar and wind prices

In [25]:
solar_df = pd.read_csv("../datasets/cleaned/solar_cost.csv")
wind_df = pd.read_csv("../datasets/cleaned/wind_cost.csv")

Here we merge the solar prices and wind prices dataframes with the state-region dataframe we created. 

Then we group by state and year and take the average price.

In [26]:
df_with_solar_prices = df.merge(solar_df, on=["Region", "Year"], how="left")
final_solar_df = df_with_solar_prices.groupby(["State", "Year"])["Price ($/MWh)"].mean().reset_index()
final_solar_df.head()

,State,Year,Price ($/MWh)
0,AL,2015,60.752500
1,AL,2016,51.520000
2,AL,2017,36.152000
3,AL,2018,28.674000
4,AL,2019,25.125556


In [27]:
df_with_wind_prices = df.merge(wind_df, on=["Region", "Year"], how="left")
final_wind_df = df_with_wind_prices.groupby(["State", "Year"])["Real 2024$/MWh"].mean().reset_index()
final_wind_df.head()

,State,Year,Real 2024$/MWh
0,AL,2015,NaN
1,AL,2016,NaN
2,AL,2017,20.0
3,AL,2018,NaN
4,AL,2019,20.0


In [28]:
final_wind_df.shape[0], final_solar_df.shape[0]

(500, 500)

We are still missing some data, but it is better

In [29]:
final_wind_df.isna().sum(), final_solar_df.isna().sum()

(State               0
 Year                0
 Real 2024$/MWh    144
 dtype: int64,
 State             0
 Year              0
 Price ($/MWh)    94
 dtype: int64)

In [30]:
final_wind_df.to_csv("../datasets/cleaned/wind_cost_per_state.csv", index = False)
final_solar_df.to_csv("../datasets/cleaned/solar_cost_per_state.csv", index = False)